# Numero di input vs Q, CQ — generazione dati

Per ogni prodotto p e ordine k, `numero_input_ordine_k` = numero di cammini di lunghezza k che arrivano in p in C
(stesso calcolo di `numero_cammini.ipynb`, C classica).
Qui si calcola la correlazione (Pearson e Spearman, sui prodotti) fra `numero_input_ordine_k` e Q, CQ
per i due algoritmi con import, a 4 e 6 digits, con α = `compute_alpha(C)`.

Output: `results/numero_cammini/correlazione_input_Q/{ALGO}/corr_input_Q_{res}_digits.parquet`

In [1]:
import sys
import numpy as np
import pandas as pd
import scipy.sparse as sp
from pathlib import Path
from scipy.stats import pearsonr, spearmanr

ROOT = next(p for p in [Path().resolve(), *Path().resolve().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

from src.algorithms_with_imports import compute_extended_fitness_with_imports, compute_extended_eci_with_imports
from src.algorithms import compute_alpha

In [2]:
DATA = ROOT / "data" / "HS02_all_years"
OUT_CAMMINI = ROOT / "results" / "numero_cammini"
OUT_CORR = OUT_CAMMINI / "correlazione_input_Q"

RESOLUTIONS = [4, 6]
K_MAX = 9
ALGOS = {
    "extended_fitness_with_imports": compute_extended_fitness_with_imports,
    "extended_eci_with_imports":     compute_extended_eci_with_imports,
}

def load_data(res):
    b = DATA / f"{res}_digits" / "alligned" / "binarized"
    Mcp_export = pd.read_parquet(b / "Mcp_export" / "Mcp_export_alligned_Y2002.parquet")
    Mcp_import = pd.read_parquet(b / "Mcp_import" / "Mcp_import_alligned_Y2002.parquet")
    C = pd.read_parquet(b / "C" / f"C_{res}_digits_HS02.parquet")
    return Mcp_export, Mcp_import, C

## Numero di input per prodotto

Se `cammini_per_product_{res}_digits.parquet` esiste lo legge, altrimenti lo calcola (come in `numero_cammini.ipynb`) e lo salva.

In [3]:
def numero_input(C, k_max=K_MAX):
    """v_k = 1^T C^k: numero di cammini di lunghezza k che arrivano in ogni prodotto."""
    C_sparse = sp.csr_matrix(C.values.astype(float))
    v = np.ones(C_sparse.shape[0])
    out = {}
    for k in range(1, k_max + 1):
        v = v @ C_sparse
        out[f"numero_input_ordine_{k}"] = np.asarray(v).ravel()
    df = pd.DataFrame(out, index=C.columns)
    df.index.name = "prodotto"
    return df.reset_index()

def load_or_compute_cammini(res, C):
    f = OUT_CAMMINI / f"cammini_per_product_{res}_digits.parquet"
    if f.exists():
        return pd.read_parquet(f)
    df = numero_input(C)
    df.to_parquet(f)
    print(f"salvato {f.relative_to(ROOT)}")
    return df

## Correlazioni

In [4]:
OUT_CORR.mkdir(parents=True, exist_ok=True)

for res in RESOLUTIONS:
    Mcp_export, Mcp_import, C = load_data(res)
    cammini = load_or_compute_cammini(res, C).set_index("prodotto")
    alpha = compute_alpha(C)

    for algo_name, algo in ALGOS.items():
        complexity, fitness, input_complexity, intrinsic_complexity, import_penalty = algo(
            Mcp_export, Mcp_import, C.T, alpha=alpha, tensor_window=3, max_iter=10000,
            tol=1e-6, track_convergence=False, save_delta=False, delta_window=None)

        Q = complexity.reindex(cammini.index)
        CQ = input_complexity.reindex(cammini.index)

        rows = []
        for col in cammini.columns:
            n_input = cammini[col]
            rows.append({
                "ordine":      int(col.rsplit("_", 1)[1]),
                "pearson_Q":   pearsonr(n_input, Q)[0],
                "spearman_Q":  spearmanr(n_input, Q)[0],
                "pearson_CQ":  pearsonr(n_input, CQ)[0],
                "spearman_CQ": spearmanr(n_input, CQ)[0],
            })
        table = pd.DataFrame(rows)
        (OUT_CORR / algo_name).mkdir(exist_ok=True)
        table.to_parquet(OUT_CORR / algo_name / f"corr_input_Q_{res}_digits.parquet")
        print(f"{res} digits — {algo_name}: fatto")

4 digits — extended_fitness_with_imports: fatto
4 digits — extended_eci_with_imports: fatto
salvato results\numero_cammini\cammini_per_product_6_digits.parquet
6 digits — extended_fitness_with_imports: fatto
6 digits — extended_eci_with_imports: fatto
